In [15]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [16]:
from datasets import load_dataset

ds = load_dataset("b-mc2/sql-create-context")["train"].shuffle(seed=42)
test = list(ds.select(range(300)))        # held-out, never trained on
train = ds.select(range(300, len(ds)))
print(len(train), "train |", len(test), "test")
print(test[0])

78277 train | 300 test
{'answer': 'SELECT venue FROM table_name_50 WHERE away_team = "essendon"', 'question': 'When Essendon played away; where did they play?', 'context': 'CREATE TABLE table_name_50 (venue VARCHAR, away_team VARCHAR)'}


In [17]:
import re, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")

SYSTEM = "You are a SQL expert. Given a table schema and a question, reply with only the SQL query."

def make_prompt(ex):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate(model, exs, bs=8):
    outs = []
    for i in range(0, len(exs), bs):
        batch = exs[i:i + bs]
        enc = tok([make_prompt(e) for e in batch], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=128, do_sample=False)
        outs += tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return outs

def strict_clean(s):
    s = re.sub(r"```(?:sql)?", "", s).strip()
    s = s.split(";")[0]
    return re.sub(r"\s+", " ", s).strip().lower()

def norm_sql(s):
    s = re.sub(r"```(?:sql)?", "", s).strip()
    if len(s) > 1 and s[0] in "\"'" and s[-1] == s[0] and s[1:].lower().startswith("select"):
        s = s[1:-1]
    s = s.split(";")[0].replace('"', "'")
    s = re.sub(r"\s+", " ", s).strip().lower()
    s = s.replace("inner join", "join")
    if " where " in s:
        head, cond = s.split(" where ", 1)
        s = head + " where " + " and ".join(sorted(p.strip() for p in cond.split(" and ")))
    return s

def score(preds, exs):
    strict = sum(strict_clean(p) == strict_clean(e["answer"]) for p, e in zip(preds, exs)) / len(exs)
    norm = sum(norm_sql(p) == norm_sql(e["answer"]) for p, e in zip(preds, exs)) / len(exs)
    return round(strict, 3), round(norm, 3)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [18]:
from google.colab import files

preds = generate(model, test)
print("Base (strict, normalized):", score(preds, test))
rows = [{"question": e["question"], "gold": e["answer"], "pred": p} for e, p in zip(test, preds)]
json.dump(rows, open("base_preds.json", "w"), indent=1)
files.download("base_preds.json")

Base (strict, normalized): (0.023, 0.57)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
N_TRAIN = 5000
sub = train.select(range(N_TRAIN))
tok.padding_side = "right"

def build(ex):
    p_ids = tok(make_prompt(ex), add_special_tokens=False)["input_ids"]
    a_ids = tok(ex["answer"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:512]
    labels = ([-100] * len(p_ids) + a_ids)[:512]
    return {"input_ids": ids, "labels": labels, "attention_mask": [1] * len(ids)}

tok_ds = sub.map(build, remove_columns=sub.column_names)
print(len(tok_ds), "training examples")

5000 training examples


In [20]:
import shutil
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

model = prepare_model_for_kbit_training(model)
cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="out",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=25,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(
    model=model, args=args, train_dataset=tok_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)
model.config.use_cache = False
trainer.train()

model.save_pretrained("qlora-adapter")
shutil.make_archive("qlora-adapter", "zip", "qlora-adapter")
files.download("qlora-adapter.zip")
print("adapter saved and downloaded")

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


Step,Training Loss
25,0.212342
50,0.085854
75,0.060546
100,0.053686
125,0.051556
150,0.044770
175,0.036174
200,0.044015
225,0.049800
250,0.037877


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

adapter saved and downloaded


In [21]:
model.config.use_cache = True
model.eval()
tok.padding_side = "left"

ft_preds = generate(model, test)
print("Base       (strict, normalized):", score(preds, test))
print("Fine-tuned (strict, normalized):", score(ft_preds, test))

rows = [{"question": e["question"], "gold": e["answer"], "pred": p} for e, p in zip(test, ft_preds)]
json.dump(rows, open("ft_preds.json", "w"), indent=1)
files.download("ft_preds.json")

Base       (strict, normalized): (0.023, 0.57)
Fine-tuned (strict, normalized): (0.767, 0.773)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from huggingface_hub import notebook_login
notebook_login()
model.push_to_hub("Varshitha2603/qwen2.5-coder-3b-text2sql-qlora")
tok.push_to_hub("Varshitha2603/qwen2.5-coder-3b-text2sql-qlora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  120MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6fw9zjdh/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/Varshitha2603/qwen2.5-coder-3b-text2sql-qlora/commit/0e28aba9029870fe9f63cd1d48a7f3058ad19fb0', commit_message='Upload tokenizer', commit_description='', oid='0e28aba9029870fe9f63cd1d48a7f3058ad19fb0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Varshitha2603/qwen2.5-coder-3b-text2sql-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Varshitha2603/qwen2.5-coder-3b-text2sql-qlora'), pr_revision=None, pr_num=None)

In [23]:
misses = [(e, p) for e, p in zip(test, ft_preds) if norm_sql(p) != norm_sql(e["answer"])]
print(len(misses), "misses out of", len(test))
for e, p in misses[:12]:
    print("Q:   ", e["question"])
    print("GOLD:", e["answer"])
    print("PRED:", p[:200])
    print()

68 misses out of 300
Q:    Of all the contestants who got voted, what is the contestant number and name of the contestant who got least votes?
GOLD: SELECT T1.contestant_number, T1.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number = T2.contestant_number GROUP BY T1.contestant_number ORDER BY COUNT(*) LIMIT 1
PRED: SELECT T2.contestant_number, T2.contestant_name FROM votes AS T1 JOIN contestants AS T2 ON T1.contestant_number = T2.contestant_number ORDER BY COUNT(*) LIMIT 1

Q:    Find the states where have the colleges whose enrollments are less than the largest size.
GOLD: SELECT DISTINCT state FROM college WHERE enr < (SELECT MAX(enr) FROM college)
PRED: SELECT state FROM college WHERE enr < (SELECT MAX(enr) FROM college)

Q:    Where was the opponent Germany?
GOLD: SELECT location FROM table_name_96 WHERE opponenent = "germany"
PRED: SELECT location FROM table_name_96 WHERE opponent = "germany"

Q:    What is the position of tickets sold/available when t